In [1]:
# Install packages
%pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 1.4 MB/s eta 0:00:00


In [2]:
# Import packages
from langchain.messages import AIMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import os
from google.colab import userdata

In [3]:
# Connect notebook to langsmith
os.environ['LANGSMITH_TRACING'] = userdata.get('LANGSMITH_TRACING')
os.environ['LANGSMITH_ENDPOINT'] = userdata.get('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = userdata.get('LANGSMITH_PROJECT')
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')
# Test the output
userdata.get('LANGSMITH_PROJECT')

'dsr_agentic_ai'

We will start exploring the use of AI Agents using the core components of messages.

Messages are the fundamental unit of context for models in LangChain.

Here the unit is an object that contains information about:

- The role (whether system or user)
- Content (actual contents of the message)
- Metadata (response information)

They represent the input, metadata, and output of the conversation when interacting with an llm.

In [4]:
# Initialise the LLM Model
model = init_chat_model("google_genai:gemini-3.1-flash-lite")

Message content is a data payload that gets sent to and from the model with a number of attributes.

The input into the model, is a **HumanMessage** object, which represents the user's input.


The output of the model, is a **AIMessage** objecttype, which represents the model's output.

A **SystemMessage** represents an initial set of instructions that primes the model's behaviour. Use it to set the tone, define the model's role, and establish guidelines for responses.

The system message is essentially the object payload to deliver the prompt, which can be a superior way to do so compared to a simple string. This gives you more control over the primpt structure, which can be useful for some models.

In [5]:
system_prompt=SystemMessage(
        content=[
            {
                "type": "text",
                "text": """You are a medieval poet and you only answer in the form of limericks.""",
            },
            {
                "type": "text",
                "text": "Specifically a Viking poet",
                "cache_control": {"type": "ephemeral"} # Tells Anthropic models to cache content block reducing latency and costs for repeated requests
            }
        ]
    )

messages = [
    system_prompt,
    HumanMessage("What is the capital of France?")
]


In [6]:
# Interact with the LLM Model
response = model.invoke(messages)
print(response)

content=[{'type': 'text', 'text': 'Across the gray waves we will steer,\nTo Paris, where drinking is dear.\nOn the Seine stands the hall,\nWhere the Franks heed the call,\nAnd leave us with naught but our beer.', 'extras': {'signature': 'EjQKMgEMOdbHHUwBEEab2PLLqckY/0ObbIQa0JcHLC3YlhayI3FO6dxF+VYCkV5iJiJHrq7l'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e355d-bd87-7e43-9173-439e97c6407c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 29, 'output_tokens': 43, 'total_tokens': 72, 'input_token_details': {'cache_read': 0}}


In [7]:
print(f"Response contents: {response.content_blocks}")
print(f"Data type of the response conents: {type(response.content_blocks)}")
print("")
print(f"Data type of the first layer of the response contents: {type(response.content_blocks[0])}")
print("")
print(response.content_blocks[0]["text"])

Response contents: [{'type': 'text', 'text': 'Across the gray waves we will steer,\nTo Paris, where drinking is dear.\nOn the Seine stands the hall,\nWhere the Franks heed the call,\nAnd leave us with naught but our beer.', 'extras': {'signature': 'EjQKMgEMOdbHHUwBEEab2PLLqckY/0ObbIQa0JcHLC3YlhayI3FO6dxF+VYCkV5iJiJHrq7l'}}]
Data type of the response conents: <class 'list'>

Data type of the first layer of the response contents: <class 'dict'>

Across the gray waves we will steer,
To Paris, where drinking is dear.
On the Seine stands the hall,
Where the Franks heed the call,
And leave us with naught but our beer.


In [8]:
# More metadata fun
print(response.usage_metadata)

{'input_tokens': 29, 'output_tokens': 43, 'total_tokens': 72, 'input_token_details': {'cache_read': 0}}


### **Parameters, parameters, and more darn parameters**

We can set the behaviour of our model at initialisation.

Some parameters  of the **init_chat_model** function we will look at are:

**temperature:** Between 0-2.This controls the randomness of the model output. The lower the number more deterministic.

**max_tokens:** Limits the number of tokens in the response. Mo' Tokens Mo' Money

**timeout:** The maximum time (in seconds) to wait for a response.

**max_retries**: Maximum time (in seconds) to wait for a response from the model before canceling request.

**Exercise:** Experiment with some of the parameters to control the output of the model.

Get really creative!

In [ ]:
# Initialise the LLM Model
model = init_chat_model(model = "google_genai:gemini-3.1-flash-lite",
                        temperature = 2,
                        max_tokens = 200)
system_msg = SystemMessage("""You are a caveman and only respond in short terse expressions.""")

messages = [
    system_msg,
    HumanMessage("What is the cause of economic inequality?")
]

# Interact with the LLM Model
response = model.invoke(messages)
print(response.content_blocks[0]["text"])

Strong man take much. Weak man get none. Cave empty.


## **init_chat_model and create_agent**
What we just used was a simple model intialiser,  which returns a preconfigured chat model isntance with no tools, loops.

create_agent() builds an agent runtime that supports:


*   Static or dynamic prompts
*   Static or dynamic tools
*   Middleware
*   Checkpointing, memory





**Create an agent with dynamic prompting**

In [9]:
# Import packages
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest

# Set the dynamic prompt using @dynamic_prompt decorator
@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")

    base_prompt = "You are a helpful assistant." # Your foundational prompt regardless of context.

    if user_role == "experienced":
        return f"{base_prompt} Explain it to me like an adult"

    elif user_role == "beginner":
        return f"{base_prompt} Explain it to me like I'm 5 years old."

    return base_prompt

In [21]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[user_role_prompt]
)

# The system prompt will be set dynamically based on context
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the cause of economic inequality?"}]},
    context={"user_role":"experienced"}
)

In [22]:
# View results
print(response)

{'messages': [HumanMessage(content='What is the cause of economic inequality?', additional_kwargs={}, response_metadata={}, id='dc2851a6-8c0e-4b66-9ccc-63ca323623d8'), AIMessage(content=[{'type': 'text', 'text': 'Economic inequality is not the result of a single factor; rather, it is the product of a complex interplay between historical legacies, technological shifts, policy choices, and global economic forces.\n\nTo understand it as an adult, it is helpful to categorize the causes into four primary drivers:\n\n### 1. Technological Change and Skill-Biased Growth\nWe live in an era where technology disproportionately rewards high-skill labor. Automation and software have replaced many "middle-skill" routine jobs (manufacturing, clerical work), pushing those workers into lower-paying service roles. Simultaneously, those who possess the skills to build, manage, or leverage these technologies capture a massive share of the economic value created. This is often referred to as **skill-biased

In [23]:
ai_messages = [
    m for m in response["messages"]
    if isinstance(m, AIMessage)
]
#print(ai_messages)
output_list=[message.content for message in ai_messages]
output_list

[[{'type': 'text',
   'text': 'Economic inequality is not the result of a single factor; rather, it is the product of a complex interplay between historical legacies, technological shifts, policy choices, and global economic forces.\n\nTo understand it as an adult, it is helpful to categorize the causes into four primary drivers:\n\n### 1. Technological Change and Skill-Biased Growth\nWe live in an era where technology disproportionately rewards high-skill labor. Automation and software have replaced many "middle-skill" routine jobs (manufacturing, clerical work), pushing those workers into lower-paying service roles. Simultaneously, those who possess the skills to build, manage, or leverage these technologies capture a massive share of the economic value created. This is often referred to as **skill-biased technological change**.\n\n### 2. The Erosion of Labor Power\nOver the past several decades, the bargaining power of the average worker has diminished significantly. This is due to 

In [24]:
# If you want a neater output
print(output_list[0][0]["text"])

Economic inequality is not the result of a single factor; rather, it is the product of a complex interplay between historical legacies, technological shifts, policy choices, and global economic forces.

To understand it as an adult, it is helpful to categorize the causes into four primary drivers:

### 1. Technological Change and Skill-Biased Growth
We live in an era where technology disproportionately rewards high-skill labor. Automation and software have replaced many "middle-skill" routine jobs (manufacturing, clerical work), pushing those workers into lower-paying service roles. Simultaneously, those who possess the skills to build, manage, or leverage these technologies capture a massive share of the economic value created. This is often referred to as **skill-biased technological change**.

### 2. The Erosion of Labor Power
Over the past several decades, the bargaining power of the average worker has diminished significantly. This is due to several structural factors:
*   **De-un